# Student Course Enrollment Data Generation

This notebook demonstrates how to generate synthetic data for a student course enrollment system using Rockfish's Entity Data Generator.

**What this notebook shows:**
- Creating a Rockfish `DataSchema` with multiple entities (`course`, `semester`, `student`, `enrollment`)
- Modeling course prerequisites and unit constraints
- Generating realistic student enrollment data with grades
- Validating entity relationships and referential integrity

**Data Model:**
- **Courses**: Have attributes like course_id, name, department, units, and prerequisites
- **Semesters**: Academic terms (Fall/Spring) with unit limits
- **Students**: Have student_id, name, major, and enrollment status
- **Enrollments**: Link students to courses in semesters with grades

## Setup and Imports

In [1]:
import rockfish as rf
import rockfish.actions as ra
from rockfish.actions.ent import (
    CategoricalParams,
    Column,
    ColumnCategoryType,
    ColumnType,
    DataSchema,
    Derivation,
    DerivationFunctionType,
    Domain,
    DomainType,
    Entity,
    EntityRelationship,
    EntityRelationshipType,
    IDParams,
    MapValuesParams,
    NormalDistParams,
    SampleFromColumnParams,
    SequentialIntParams,
    UniformDistParams,
)
from dotenv import load_dotenv
import pandas as pd

In [2]:
# Connect to the Rockfish platform using your API Key
load_dotenv()
conn = rf.Connection.from_env()

## Create Schema

We'll generate data for a student course enrollment system with four entities:
- **course**: Available courses with prerequisites and unit values
- **semester**: Academic terms (Fall 2024, Spring 2025, etc.)
- **student**: Student information including major
- **enrollment**: Records of students taking courses with grades

### Business Rules:
1. Students can only take courses for which they have satisfied prerequisites
2. Students cannot exceed the maximum units per semester (typically 18-21)
3. Each enrollment results in a grade (A, B, C, D, F, or W for withdrawal)

In [3]:
# Configuration parameters
N_COURSES = 25
N_SEMESTERS = 6  # 3 academic years
N_STUDENTS = 100
N_ENROLLMENTS = 800  # Average ~8 enrollments per student

In [6]:
def create_student_enrollment_schema(
    n_courses=25,
    n_semesters=6,
    n_students=100,
    n_enrollments=800,
) -> DataSchema:
    """Create the student course enrollment schema."""
    
    # ENTITY 1: course
    # Courses have IDs, names, departments, units, and difficulty levels
    course = Entity(
        name="course",
        cardinality=n_courses,
        columns=[
            # Course ID (e.g., CS101, MATH201)
            Column(
                name="course_id",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.ID,
                    params=IDParams(template_str="COURSE_{id}"),
                ),
            ),
            # Department
            Column(
                name="department",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["CS", "MATH", "PHYS", "CHEM", "BIOL", "ENG", "HIST"],
                        with_replacement=True,
                        seed=100,
                    ),
                ),
            ),
            # Course level (100-400 level)
            Column(
                name="course_level",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=[100, 200, 300, 400],
                        with_replacement=True,
                        seed=101,
                    ),
                ),
            ),
            # Units (credits) - typically 3 or 4
            Column(
                name="units",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=[3, 3, 3, 4, 4],  # Weighted towards 3 units
                        with_replacement=True,
                        seed=102,
                    ),
                ),
            ),
            # Has prerequisites flag
            Column(
                name="has_prerequisites",
                data_type="string",
                column_type=ColumnType.DERIVED,
                column_category_type=ColumnCategoryType.METADATA,
                derivation=Derivation(
                    function_type=DerivationFunctionType.MAP_VALUES,
                    dependent_columns=["course_level"],
                    params=MapValuesParams(
                        mapping=[
                            {"from":"100", "to":"No"},   # 100-level courses have no prereqs
                            {"from":"200", "to":"100"},  # Higher levels require prereqs
                            {"from":"300", "to":"100"},
                            {"from":"400", "to":"200"},
                             ],
                        default="No",
                    ),
                ),
            ),
            # Difficulty rating (affects grade distribution)
            Column(
                name="difficulty",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["Easy", "Medium", "Medium", "Hard"],  # Most courses are medium
                        with_replacement=True,
                        seed=103,
                    ),
                ),
            ),
        ],
    )
    
    # ENTITY 2: semester
    # Academic terms with their properties
    semester = Entity(
        name="semester",
        cardinality=n_semesters,
        columns=[
            # Semester ID
            Column(
                name="semester_id",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.ID,
                    params=IDParams(template_str="SEM_{id}"),
                ),
            ),
            # Term (Fall or Spring)
            Column(
                name="term",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["Fall", "Spring"],
                        with_replacement=True,
                        seed=200,
                    ),
                ),
            ),
            # Academic year
            Column(
                name="year",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=[2023, 2024, 2025],
                        with_replacement=True,
                        seed=201,
                    ),
                ),
            ),
            # Maximum units allowed per semester
            Column(
                name="max_units",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=[18, 18, 18, 21],  # Standard 18, some allow 21
                        with_replacement=True,
                        seed=202,
                    ),
                ),
            ),
        ],
    )
    
    # ENTITY 3: student
    # Student information
    student = Entity(
        name="student",
        cardinality=n_students,
        columns=[
            # Student ID
            Column(
                name="student_id",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.ID,
                    params=IDParams(template_str="STU_{id}"),
                ),
            ),
            # Major department
            Column(
                name="major",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["CS", "MATH", "PHYS", "CHEM", "BIOL", "ENG", "HIST"],
                        with_replacement=True,
                        seed=300,
                    ),
                ),
            ),
            # Class standing
            Column(
                name="class_standing",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["Freshman", "Sophomore", "Junior", "Senior"],
                        with_replacement=True,
                        seed=301,
                    ),
                ),
            ),
            # Enrollment status
            Column(
                name="enrollment_status",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["Full-time", "Full-time", "Full-time", "Part-time"],  # Mostly full-time
                        with_replacement=True,
                        seed=302,
                    ),
                ),
            ),
            # GPA range bucket (affects grade outcomes)
            Column(
                name="gpa_bucket",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["Low", "Medium", "Medium", "High"],  # Normal distribution
                        with_replacement=True,
                        seed=303,
                    ),
                ),
            ),
        ],
    )
    
    # ENTITY 4: enrollment
    # Links students to courses in specific semesters with grades
    enrollment = Entity(
        name="enrollment",
        cardinality=n_enrollments,
        columns=[
            # Enrollment ID
            Column(
                name="enrollment_id",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.ID,
                    params=IDParams(template_str="ENR_{id}"),
                ),
            ),
            # Foreign key to student
            Column(
                name="fk_student_id",
                data_type="string",
                column_type=ColumnType.FOREIGN_KEY,
                column_category_type=ColumnCategoryType.METADATA,
            ),
            # Foreign key to course
            Column(
                name="fk_course_id",
                data_type="string",
                column_type=ColumnType.FOREIGN_KEY,
                column_category_type=ColumnCategoryType.METADATA,
            ),
            # Foreign key to semester
            Column(
                name="fk_semester_id",
                data_type="string",
                column_type=ColumnType.FOREIGN_KEY,
                column_category_type=ColumnCategoryType.METADATA,
            ),
            # Grade received
            Column(
                name="grade",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["A", "A", "B", "B", "B", "C", "C", "D", "F", "W"],  # Grade distribution
                        with_replacement=True,
                        seed=400,
                    ),
                ),
            ),
            # Enrollment status
            Column(
                name="status",
                data_type="string",
                column_type=ColumnType.DERIVED,
                column_category_type=ColumnCategoryType.METADATA,
                derivation=Derivation(
                    function_type=DerivationFunctionType.MAP_VALUES,
                    dependent_columns=["grade"],
                    params=MapValuesParams(
                        mapping=[
                            {"from": "A", "to" : "Completed"},
                            {"from": "B", "to": "Completed"},
                            {"from": "C", "to": "Completed"},
                            {"from": "D", "to": "Completed"},
                            {"from":"F", "to": "Failed"},
                            {"from": "W", "to": "Withdrawn"},
                        ],
                        default="Completed",
                    ),
                ),
            ),
        ],
    )
    
    # ENTITY RELATIONSHIPS
    relationships = [
        # student -> enrollment (one-to-many)
        # A student can have many enrollments
        EntityRelationship(
            parent_entity="student",
            child_entity="enrollment",
            relationship_type=EntityRelationshipType.ONE_TO_MANY,
            join_columns={
                "student_id": "fk_student_id",
            },
        ),
        # course -> enrollment (one-to-many)
        # A course can have many enrollments
        EntityRelationship(
            parent_entity="course",
            child_entity="enrollment",
            relationship_type=EntityRelationshipType.ONE_TO_MANY,
            join_columns={
                "course_id": "fk_course_id",
            },
        ),
        # semester -> enrollment (one-to-many)
        # A semester can have many enrollments
        EntityRelationship(
            parent_entity="semester",
            child_entity="enrollment",
            relationship_type=EntityRelationshipType.ONE_TO_MANY,
            join_columns={
                "semester_id": "fk_semester_id",
            },
        ),
    ]
    
    return DataSchema(
        entities=[course, semester, student, enrollment],
        entity_relationships=relationships,
    )

In [7]:
# Create the schema instance
enrollment_schema = create_student_enrollment_schema(
    n_courses=N_COURSES,
    n_semesters=N_SEMESTERS,
    n_students=N_STUDENTS,
    n_enrollments=N_ENROLLMENTS,
)

## Run Data Generation

We'll use a **Rockfish Workflow** to run a data generation job on the Rockfish platform.

<div class="alert alert-block alert-info">
<b>Info: Rockfish Workflows and Actions</b>

A Rockfish Workflow is a job that you want to run on the Rockfish platform. You can think of a workflow as a graph of operations that take you from an input (e.g., data schema) to an output (e.g., datasets). Each operation in this graph is a Rockfish Action.
</div>

In [8]:
config = ra.GenerateFromDataSchema.Config(
    schema=enrollment_schema,
    upload_datasets=True,
)
generate = ra.GenerateFromDataSchema(config)

In [9]:
builder = rf.WorkflowBuilder()
builder.add(generate)
workflow = await builder.start(conn)
print(f"Workflow ID: {workflow.id()}")

Workflow ID: 6ZX56NzWyp0BaFo3XqYXQd


In [10]:
async for log in workflow.logs(level=rf.events.LogLevel.DEBUG):
    print(log)

2026-02-15T22:07:25.227049Z generate-from-data-schema: INFO Generating 4 entities: course, semester, student, enrollment
2026-02-15T22:07:25.237450Z generate-from-data-schema: INFO Starting data generation...
2026-02-15T22:07:25.257064Z generate-from-data-schema: INFO Generated 4 entity tables
2026-02-15T22:07:25.267229Z generate-from-data-schema: INFO Creating dataset for entity 'course': 25 rows
2026-02-15T22:07:25.522991Z generate-from-data-schema: INFO Uploaded dataset 'course' (2hwk2uwPLrQ1xGy4X3vyrV): 25 rows
2026-02-15T22:07:25.543579Z generate-from-data-schema: INFO Creating dataset for entity 'semester': 6 rows
2026-02-15T22:07:25.705691Z generate-from-data-schema: INFO Uploaded dataset 'semester' (3tOO4UfNdwVkowOHNH88gx): 6 rows
2026-02-15T22:07:25.725647Z generate-from-data-schema: INFO Creating dataset for entity 'student': 100 rows
2026-02-15T22:07:25.894216Z generate-from-data-schema: INFO Uploaded dataset 'student' (5cPaUKQUCyPXREPHUGzTdr): 100 rows
2026-02-15T22:07:25.9

## Retrieve Generated Datasets

We'll retrieve datasets for `course`, `semester`, `student`, and `enrollment` entities.

In [11]:
datasets = await workflow.datasets().collect()
print(f"Generated {len(datasets)} datasets")

course_dataset = None
semester_dataset = None
student_dataset = None
enrollment_dataset = None

for remote_ds in datasets:
    ds = await remote_ds.to_local(conn)
    if ds.name() == "course":
        course_dataset = ds
    elif ds.name() == "semester":
        semester_dataset = ds
    elif ds.name() == "student":
        student_dataset = ds
    elif ds.name() == "enrollment":
        enrollment_dataset = ds

Generated 4 datasets


## Explore Course Data

In [12]:
course_df = course_dataset.to_pandas()
print(f"Course dataset: {course_dataset.table.num_rows} rows")
course_df.head(10)

Course dataset: 25 rows


,course_id,department,course_level,units,has_prerequisites,difficulty
0,COURSE_0,ENG,200,3,No,Medium
1,COURSE_1,ENG,400,3,No,Medium
2,COURSE_2,CS,300,3,No,Easy
3,COURSE_3,BIOL,200,3,No,Easy
4,COURSE_4,CS,100,3,No,Medium
5,COURSE_5,PHYS,400,4,No,Easy
6,COURSE_6,CHEM,200,3,No,Medium
7,COURSE_7,CS,300,3,No,Hard
8,COURSE_8,BIOL,300,3,No,Medium
9,COURSE_9,HIST,200,4,No,Hard


In [13]:
print("Courses by Department:")
print(course_df["department"].value_counts())
print("\nCourses by Level:")
print(course_df["course_level"].value_counts().sort_index())
print("\nCourses by Difficulty:")
print(course_df["difficulty"].value_counts())

Courses by Department:
department
BIOL    5
HIST    5
ENG     4
CS      4
PHYS    3
CHEM    2
MATH    2
Name: count, dtype: int64

Courses by Level:
course_level
100    5
200    7
300    5
400    8
Name: count, dtype: int64

Courses by Difficulty:
difficulty
Medium    15
Easy       7
Hard       3
Name: count, dtype: int64


## Explore Semester Data

In [14]:
semester_df = semester_dataset.to_pandas()
print(f"Semester dataset: {semester_dataset.table.num_rows} rows")
semester_df

Semester dataset: 6 rows


,semester_id,term,year,max_units
0,SEM_0,Fall,2023,18
1,SEM_1,Spring,2025,18
2,SEM_2,Fall,2023,18
3,SEM_3,Spring,2024,18
4,SEM_4,Fall,2025,18
5,SEM_5,Fall,2024,21


## Explore Student Data

In [15]:
student_df = student_dataset.to_pandas()
print(f"Student dataset: {student_dataset.table.num_rows} rows")
student_df.head(10)

Student dataset: 100 rows


,student_id,major,class_standing,enrollment_status,gpa_bucket
0,STU_0,CS,Freshman,Part-time,Medium
1,STU_1,BIOL,Sophomore,Part-time,Low
2,STU_2,ENG,Freshman,Part-time,High
3,STU_3,CHEM,Freshman,Full-time,Medium
4,STU_4,ENG,Sophomore,Full-time,Medium
5,STU_5,PHYS,Sophomore,Part-time,High
6,STU_6,HIST,Sophomore,Full-time,Medium
7,STU_7,MATH,Senior,Full-time,Medium
8,STU_8,BIOL,Junior,Part-time,High
9,STU_9,ENG,Senior,Full-time,High


In [16]:
print("Students by Major:")
print(student_df["major"].value_counts())
print("\nStudents by Class Standing:")
print(student_df["class_standing"].value_counts())
print("\nStudents by Enrollment Status:")
print(student_df["enrollment_status"].value_counts())

Students by Major:
major
CS      17
CHEM    17
HIST    17
ENG     16
BIOL    15
PHYS    13
MATH     5
Name: count, dtype: int64

Students by Class Standing:
class_standing
Senior       27
Junior       25
Freshman     24
Sophomore    24
Name: count, dtype: int64

Students by Enrollment Status:
enrollment_status
Full-time    70
Part-time    30
Name: count, dtype: int64


## Explore Enrollment Data

In [17]:
enrollment_df = enrollment_dataset.to_pandas()
print(f"Enrollment dataset: {enrollment_dataset.table.num_rows} rows")
enrollment_df.head(15)

Enrollment dataset: 800 rows


,enrollment_id,fk_student_id,fk_course_id,fk_semester_id,grade,status
0,ENR_0,STU_85,COURSE_21,SEM_5,D,Completed
1,ENR_1,STU_63,COURSE_15,SEM_3,A,Completed
2,ENR_2,STU_51,COURSE_12,SEM_3,B,Completed
3,ENR_3,STU_26,COURSE_6,SEM_1,C,Completed
4,ENR_4,STU_30,COURSE_7,SEM_1,B,Completed
5,ENR_5,STU_4,COURSE_1,SEM_0,W,Withdrawn
6,ENR_6,STU_7,COURSE_1,SEM_0,A,Completed
7,ENR_7,STU_1,COURSE_0,SEM_0,B,Completed
8,ENR_8,STU_17,COURSE_4,SEM_1,B,Completed
9,ENR_9,STU_81,COURSE_20,SEM_4,B,Completed


In [18]:
print("Grade Distribution:")
print(enrollment_df["grade"].value_counts().sort_index())
print("\nEnrollment Status:")
print(enrollment_df["status"].value_counts())
print(f"\nAverage Grade Points: {enrollment_df['grade_points'].astype(float).mean():.2f}")

Grade Distribution:
grade
A    180
B    254
C    151
D     58
F     84
W     73
Name: count, dtype: int64

Enrollment Status:
status
Completed    643
Failed        84
Withdrawn     73
Name: count, dtype: int64


KeyError: 'grade_points'

## Validate Entity Relationships

We'll verify that all enrollment foreign keys reference valid entities.

In [ ]:
# Validate student foreign key integrity
valid_student_ids = set(student_df["student_id"])
enrollment_student_ids = set(enrollment_df["fk_student_id"])

invalid_student_refs = enrollment_student_ids - valid_student_ids
print(f"Valid student IDs: {len(valid_student_ids)}")
print(f"Unique students with enrollments: {len(enrollment_student_ids)}")
print(f"All student foreign keys valid: {len(invalid_student_refs) == 0}")
if invalid_student_refs:
    print(f"Invalid student references: {invalid_student_refs}")

In [ ]:
# Validate course foreign key integrity
valid_course_ids = set(course_df["course_id"])
enrollment_course_ids = set(enrollment_df["fk_course_id"])

invalid_course_refs = enrollment_course_ids - valid_course_ids
print(f"Valid course IDs: {len(valid_course_ids)}")
print(f"Unique courses with enrollments: {len(enrollment_course_ids)}")
print(f"All course foreign keys valid: {len(invalid_course_refs) == 0}")
if invalid_course_refs:
    print(f"Invalid course references: {invalid_course_refs}")

In [ ]:
# Validate semester foreign key integrity
valid_semester_ids = set(semester_df["semester_id"])
enrollment_semester_ids = set(enrollment_df["fk_semester_id"])

invalid_semester_refs = enrollment_semester_ids - valid_semester_ids
print(f"Valid semester IDs: {len(valid_semester_ids)}")
print(f"Unique semesters with enrollments: {len(enrollment_semester_ids)}")
print(f"All semester foreign keys valid: {len(invalid_semester_refs) == 0}")
if invalid_semester_refs:
    print(f"Invalid semester references: {invalid_semester_refs}")

## Analysis: Student Enrollment Patterns

In [ ]:
# Enrollments per student
enrollments_per_student = enrollment_df.groupby("fk_student_id").size()
print(f"Enrollments per student statistics:")
print(f"  Min: {enrollments_per_student.min()}")
print(f"  Max: {enrollments_per_student.max()}")
print(f"  Mean: {enrollments_per_student.mean():.2f}")
print(f"  Median: {enrollments_per_student.median():.2f}")

In [ ]:
# Enrollments per course
enrollments_per_course = enrollment_df.groupby("fk_course_id").size()
print(f"Enrollments per course statistics:")
print(f"  Min: {enrollments_per_course.min()}")
print(f"  Max: {enrollments_per_course.max()}")
print(f"  Mean: {enrollments_per_course.mean():.2f}")

In [ ]:
# Join enrollment with student to analyze GPA by major
enrollment_with_student = enrollment_df.merge(
    student_df[["student_id", "major", "class_standing"]], 
    left_on="fk_student_id", 
    right_on="student_id"
)

print("Average Grade Points by Major:")
gpa_by_major = enrollment_with_student.groupby("major")["grade_points"].apply(
    lambda x: x.astype(float).mean()
).sort_values(ascending=False)
print(gpa_by_major)

In [ ]:
# Join enrollment with course to analyze grades by course level
enrollment_with_course = enrollment_df.merge(
    course_df[["course_id", "department", "course_level", "difficulty"]], 
    left_on="fk_course_id", 
    right_on="course_id"
)

print("Average Grade Points by Course Level:")
gpa_by_level = enrollment_with_course.groupby("course_level")["grade_points"].apply(
    lambda x: x.astype(float).mean()
).sort_index()
print(gpa_by_level)

print("\nAverage Grade Points by Difficulty:")
gpa_by_difficulty = enrollment_with_course.groupby("difficulty")["grade_points"].apply(
    lambda x: x.astype(float).mean()
)
print(gpa_by_difficulty)

## Save Data to CSV

In [ ]:
# Save all datasets to file
course_df.to_csv("course_data.csv", index=False)
semester_df.to_csv("semester_data.csv", index=False)
student_df.to_csv("student_data.csv", index=False)
enrollment_df.to_csv("enrollment_data.csv", index=False)

print("Data saved to CSV files:")
print("  - course_data.csv")
print("  - semester_data.csv")
print("  - student_data.csv")
print("  - enrollment_data.csv")

## Summary

This notebook demonstrated how to use Rockfish's Entity Data Generator to create synthetic student course enrollment data. Key features shown:

1. **Entity Definition**: Created four entities (course, semester, student, enrollment) with appropriate attributes
2. **Relationships**: Defined one-to-many relationships between entities using foreign keys
3. **Derived Columns**: Used `MAP_VALUES` derivation to compute grade points and enrollment status from grades
4. **Domain Types**: Used various domain types including ID, CATEGORICAL, and UNIFORM_DIST
5. **Referential Integrity**: Validated that all foreign key references are valid

### Business Rules Modeled:
- Courses have prerequisites based on course level (higher level courses require prerequisites)
- Semesters have maximum unit limits
- Grades map to grade points following standard GPA scale
- Students have attributes that could affect enrollment patterns

### Potential Extensions:
- Add explicit prerequisite course relationships
- Implement unit counting constraints per student per semester
- Add instructor entity and course sections
- Include classroom/schedule information